# Sprint 007 D3 — Required execution envelope

**Question:** What fraction of the quoted half-spread can the frozen Sprint 006 selected option book afford to pay while remaining profitable and retaining meaningful economic margin?

\(h=0\) is midpoint. \(h=1\) is full cross. \(h\) is a mechanical interpolation, not expected real execution.

**Path R** (resized Tier A, by trade date) is the primary book-level envelope. **Path F** (frozen midpoint quantities) is a diagnostic of resizing and must not bind Path R.

The answer is a **range**, not one fill: below \(h_{R,50}\) retains 50% of midpoint dollar P&L; below \(h_{R,25}\) retains 25%; below \(h_{R,P0}\) remains dollar-profitable. \(h_{R,\mathrm{CAR}0}\) is companion only. There is no `h_req`.

Formulas are frozen in [`docs/tmp/sprint007_d3_design.md`](../../docs/tmp/sprint007_d3_design.md).

**Non-goals:** filters, alternative structures, commissions as a model, fill probability, `SurfaceRunner`, D4.

In [ ]:
import sys
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "backtest").is_dir() and (candidate / "setup.py").exists():
            return candidate
    raise RuntimeError("Could not locate MomentumCVG repo root (set cwd or PYTHONPATH)")


REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.backtest.sprint007_d3_execution_envelope import (
    VERDICT_BLOCKED,
    run_d3_analysis,
)

pd.set_option("display.float_format", lambda v: f"{v:,.6f}")
plt.rcParams["figure.figsize"] = (9, 4)

## 1. Preconditions

D0 readiness, D1 continue, and accepted D2 class `D3_EXECUTION_FOCUSED` are hard prerequisites. A fail is `D3_BLOCKED` — do not interpret crossings.

In [ ]:
result = run_d3_analysis()
print("D3 verdict:", result.verdict)
if result.blocked or result.verdict == VERDICT_BLOCKED:
    raise RuntimeError(f"D3 blocked: {result.blocker}")
print("official:", result.manifest.get("official"))
pd.DataFrame(result.reconciliation)

## 2. Endpoint reconciliation

Accepted calculation. Path F \(h=0\) must match official midpoint; Path F \(h=1\) the D2 hybrid \(P(Q_{\mathrm{mid}}, p_{\mathrm{cross}})\); Path R \(h=0\) official mid quantities and P&L; Path R \(h=1\) official cross. Stop if any check fails — do not approximate.

In [ ]:
recon = pd.DataFrame(result.reconciliation)
assert recon["passed"].all(), recon.loc[~recon["passed"]]
recon

## 3. Path R and Path F curves on \(H_{\mathrm{vis}}\)

21-point visualization grid only. Authoritative roots are not interpolants on this grid.

In [ ]:
curves = result.curves.copy()
curves

## 4. Path R envelope

D3 gate statistic. Primary result is the resized range, not one chosen fill.

In [ ]:
print(
    f"On the resized book, h < {result.h_R_50} retains at least 50% of midpoint dollar P&L; "
    f"h < {result.h_R_25} retains at least 25%; h < {result.h_R_P0} remains dollar-profitable. "
    f"Path R CAR first reaches zero at {result.h_R_CAR0} (companion)."
)
pd.DataFrame(
    [row.as_record() for row in result.crossings if row.path == "R"]
)

## 5. Path F diagnostics

Exploratory description. \(h_R - h_F\) shows the effect of resizing. Path F must not bind Path R.

In [ ]:
print("Path F is diagnostic only; it does not bind the Path R envelope.")
print(result.resize_gaps)
pd.DataFrame(
    [row.as_record() for row in result.crossings if row.path == "F"]
)

## 6. Path R headroom and distances to cross

Headroom is only `headroom_50_to_25` and `headroom_25_to_P0` (positive = additional execution-cost capacity). `distance_P0_to_cross` and `distance_CAR0_to_cross` measure how far full cross lies beyond break-even; they are not headroom.

In [ ]:
pd.Series(
    {
        "headroom_50_to_25": result.headroom_50_to_25,
        "headroom_25_to_P0": result.headroom_25_to_P0,
        "distance_P0_to_cross": result.distance_P0_to_cross,
        "distance_CAR0_to_cross": result.distance_CAR0_to_cross,
        "P_R_monotonic_nonincreasing": result.monotonicity.get("P_R"),
        "CAR_R_monotonic_nonincreasing": result.monotonicity.get("CAR_R"),
    }
)

## 7. Long vs short P&L snapshots

Exploratory description. Not a side-only strategy search.

In [ ]:
print("Path R long/short P&L at exact roots (not H_vis).")
result.side_snapshot

## 8. Visualizations

Path R solid, Path F dashed. Mark Path R dollar thresholds on P&L and Path R CAR break-even on CAR. No `h_req` line.

In [ ]:
r = result.curves[result.curves["path"] == "R"]
f = result.curves[result.curves["path"] == "F"]
m0 = float(r.loc[r["h"] == 0.0, "pnl"].iloc[0])

fig, ax = plt.subplots()
ax.plot(r["h"], r["pnl"], label="Path R", color="C0")
ax.plot(f["h"], f["pnl"], label="Path F", color="C0", linestyle="--")
for y, name in ((m0, "M"), (0.5 * m0, "0.50M"), (0.25 * m0, "0.25M"), (0.0, "0")):
    ax.axhline(y, color="0.6", linewidth=0.8)
for h, name in ((result.h_R_50, "h_R,50"), (result.h_R_25, "h_R,25"), (result.h_R_P0, "h_R,P0")):
    if h is not None:
        ax.axvline(h, color="C3", linewidth=0.8, alpha=0.8)
ax.set_xlabel("h")
ax.set_ylabel("Portfolio P&L")
ax.set_title("Portfolio P&L vs h")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(r["h"], r["car"], label="Path R", color="C1")
ax.plot(f["h"], f["car"], label="Path F", color="C1", linestyle="--")
ax.axhline(0.0, color="0.6", linewidth=0.8)
if result.h_R_CAR0 is not None:
    ax.axvline(result.h_R_CAR0, color="C3", linewidth=0.8, alpha=0.8)
ax.set_xlabel("h")
ax.set_ylabel("View A mean cycle CAR")
ax.set_title("View A mean cycle CAR vs h")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(r["h"], r["sum_abs_qty"], label="Path R", color="C2")
ax.plot(f["h"], f["sum_abs_qty"], label="Path F", color="C2", linestyle="--")
ax.set_xlabel("h")
ax.set_ylabel(r"$\sum |Q|$")
ax.set_title(r"$\sum |Q|$ vs h")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(r["h"], r["capital"], label="Path R", color="C4")
ax.plot(f["h"], f["capital"], label="Path F", color="C4", linestyle="--")
ax.set_xlabel("h")
ax.set_ylabel(r"$\sum$ capital at risk")
ax.set_title(r"$\sum$ capital-at-risk vs h")
ax.legend()
plt.show()

## 9. Limits

The Path R range is a **requirement envelope**, not a live limit price and not one chosen fill. Path F remaining positive is not executable return. Historical end-of-day quotes do not show whether a package order would fill inside that envelope. Commissions, missed fills, timing, and adverse selection are unmodeled; that is why the 50% and 25% buffers exist. Distances to full cross are not headroom. D4, not this notebook, chooses the sprint outcome. Forbidden language: recoverable, likely fillable, patient execution would capture midpoint, historical quotes imply attainable package fills.